
5: Product Enhancement Recommendations and Final Report
--------------------------------------------------------------

Synthesize findings into product enhancement recommendations and prepare the final report:

1. Based on all previous analysis, recommend data-driven product changes:
   - Optimal fee structure adjustments
   - Tenure options modifications
   - Loan amount limits
   - Repayment method incentives
2. Quantify the potential impact of recommendations:
   - Expected revenue increase
   - Risk reduction potential
   - Customer experience improvements
3. Compile a comprehensive final report with:
   - Executive summary
   - Methodology and assumptions
   - Key findings from all analyses
   - Visualizations and supporting data
   - Recommendations with implementation roadmap
4. Prepare presentation-ready visualizations for leadership
5. Document all SQL queries and Python code used in the analysis

Expected output: Product enhancement recommendations with quantified impact, comprehensive final report, and presentation-ready materials.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

# Define file path
excel_file = r"C:\Users\moses_y\OneDrive\Desktop\ML Projects\BI Analyst Case study\assets\BI_Analyst_Case_Study_Data.xlsx"

print("=== PRODUCT ENHANCEMENT RECOMMENDATIONS AND FINAL REPORT ===")

# ===== DATA LOADING AND CLEANING =====
def load_and_clean_data(file_path):
    print("\nLoading and cleaning data...")
    
    # Load both sheets
    disbursements_raw = pd.read_excel(file_path, sheet_name='Disbursements')
    repayments_raw = pd.read_excel(file_path, sheet_name='Repayments')
    
    print(f"- Disbursements: {disbursements_raw.shape[0]} rows, {disbursements_raw.shape[1]} columns")
    print(f"- Repayments: {repayments_raw.shape[0]} rows, {repayments_raw.shape[1]} columns")
    
    # Clean disbursements data
    disbursements_df = disbursements_raw.copy()
    
    # Rename columns for clarity
    disbursements_df.rename(columns={
        'customer_id': 'customer_id',
        'disb_date': 'disbursement_date',
        'tenure': 'tenure_str',
        'account_num': 'account_number',
        'loan_amount': 'loan_amount',
        'loan_fee': 'loan_fee'
    }, inplace=True)
    
    # Convert date to datetime
    disbursements_df['disbursement_date'] = pd.to_datetime(disbursements_df['disbursement_date'], errors='coerce')
    
    # Extract numeric values from tenure
    def extract_tenure_days(tenure_val):
        if isinstance(tenure_val, str):
            # Extract digits only
            digits = ''.join(filter(str.isdigit, tenure_val))
            return int(digits) if digits else np.nan
        elif isinstance(tenure_val, (int, float)):
            return tenure_val
        return np.nan
    
    disbursements_df['tenure_days'] = disbursements_df['tenure_str'].apply(extract_tenure_days)
    
    # Drop rows with missing essential data
    disbursements_df.dropna(subset=['customer_id', 'disbursement_date', 'loan_amount', 'tenure_days'], inplace=True)
    
    # Convert tenure_days to integer
    disbursements_df['tenure_days'] = disbursements_df['tenure_days'].astype(int)
    
    # Calculate expected repayment date
    disbursements_df['expected_repayment_date'] = disbursements_df['disbursement_date'] + pd.to_timedelta(disbursements_df['tenure_days'], unit='D')
    
    # Calculate expected repayment amount (loan amount + fee)
    disbursements_df['expected_repayment'] = disbursements_df['loan_amount'] + disbursements_df['loan_fee']
    
    # Calculate fee percentage
    disbursements_df['fee_percentage'] = (disbursements_df['loan_fee'] / disbursements_df['loan_amount']) * 100
    
    # Extract month and year for time series analysis
    disbursements_df['month'] = disbursements_df['disbursement_date'].dt.to_period('M')
    
    # Clean repayments data
    repayments_df = repayments_raw.copy()
    
    # Rename columns for clarity
    repayments_df.rename(columns={
        'date_time': 'repayment_date_str',
        'customer_id': 'customer_id',
        'amount': 'repayment_amount',
        'rep_month': 'repayment_month',
        'repayment_type': 'repayment_type'
    }, inplace=True)
    
    # Function to parse Oracle date format
    def parse_oracle_date(date_str):
        if not isinstance(date_str, str):
            return pd.NaT
            
        try:
            # Format: '27-JUN-24 07.16.36.000000000 AM'
            parts = date_str.split(' ')
            if len(parts) != 3:
                return pd.NaT
                
            date_part = parts[0]  # '27-JUN-24'
            time_part = parts[1]  # '07.16.36.000000000'
            am_pm = parts[2]      # 'AM'
            
            # Parse date part
            day, month, year = date_part.split('-')
            month_dict = {
                'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
                'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
            }
            month_num = month_dict.get(month.upper(), 1)  # Default to January if invalid
            year_num = 2000 + int(year) if len(year) == 2 else int(year)
            
            # Parse time part
            time_parts = time_part.split('.')
            hour = int(time_parts[0])
            minute = int(time_parts[1])
            second = int(time_parts[2][:2])  # Take only first two digits of seconds
            
            # Adjust for AM/PM
            if am_pm.upper() == 'PM' and hour < 12:
                hour += 12
            elif am_pm.upper() == 'AM' and hour == 12:
                hour = 0
                
            return datetime(year_num, month_num, int(day), hour, minute, second)
        except Exception as e:
            return pd.NaT
    
    # Apply date parsing
    repayments_df['repayment_date'] = repayments_df['repayment_date_str'].apply(parse_oracle_date)
    
    # Drop rows with missing essential data
    repayments_df.dropna(subset=['customer_id', 'repayment_date', 'repayment_amount'], inplace=True)
    
    # Ensure repayment_amount is numeric
    repayments_df['repayment_amount'] = pd.to_numeric(repayments_df['repayment_amount'], errors='coerce')
    
    # Extract month from repayment date
    repayments_df['month'] = repayments_df['repayment_date'].dt.to_period('M')
    
    # Drop unnecessary columns
    repayments_df.drop('repayment_date_str', axis=1, inplace=True)
    
    print("Data cleaning complete.")
    return disbursements_df, repayments_df

# ===== PREPARE LOAN PERFORMANCE DATA =====
def prepare_loan_performance_data(disbursements_df, repayments_df):
    print("\nPreparing loan performance data...")
    
    # Prepare loan status data
    # Aggregate repayments by customer
    customer_repayments = repayments_df.groupby('customer_id')['repayment_amount'].sum().reset_index()
    customer_repayments.rename(columns={'repayment_amount': 'total_repaid'}, inplace=True)
    
    # Merge with disbursements
    loan_status = disbursements_df.merge(customer_repayments, on='customer_id', how='left')
    loan_status['total_repaid'] = loan_status['total_repaid'].fillna(0)
    
    # Calculate outstanding amounts
    loan_status['outstanding_amount'] = loan_status['expected_repayment'] - loan_status['total_repaid']
    loan_status['outstanding_amount'] = loan_status['outstanding_amount'].clip(lower=0)  # No negative outstanding
    
    # Calculate repayment ratio
    loan_status['repayment_ratio'] = loan_status['total_repaid'] / loan_status['expected_repayment']
    loan_status['repayment_ratio'] = loan_status['repayment_ratio'].clip(upper=1.0)  # Cap at 100%
    
    # Calculate days past due
    current_date = datetime.now()
    loan_status['days_past_due'] = (current_date - loan_status['expected_repayment_date']).dt.days
    
    # Identify defaulted loans (past due date with outstanding balance)
    loan_status['is_defaulted'] = (loan_status['days_past_due'] > 0) & (loan_status['outstanding_amount'] > 0)
    
    # Calculate profit per loan
    loan_status['profit'] = loan_status['total_repaid'] - loan_status['loan_amount']
    loan_status['profit_margin'] = loan_status['profit'] / loan_status['loan_amount'] * 100
    
    # Define loan amount bins
    loan_bins = [0, 500, 1000, 2000, 5000, 10000, float('inf')]
    loan_labels = ['0-500', '501-1000', '1001-2000', '2001-5000', '5001-10000', '10000+']
    
    loan_status['loan_amount_bin'] = pd.cut(loan_status['loan_amount'], bins=loan_bins, labels=loan_labels)
    
    # Define tenure bins
    tenure_bins = [0, 30, 60, 90, 180, 365, float('inf')]
    tenure_labels = ['0-30 days', '31-60 days', '61-90 days', '91-180 days', '181-365 days', '365+ days']
    
    loan_status['tenure_bin'] = pd.cut(loan_status['tenure_days'], bins=tenure_bins, labels=tenure_labels)
    
    # Analyze repayment types
    repayment_types = repayments_df.groupby('repayment_type').agg({
        'repayment_amount': 'sum',
        'customer_id': 'count'
    }).reset_index()
    
    repayment_types.rename(columns={'customer_id': 'repayment_count'}, inplace=True)
    repayment_types['percentage'] = repayment_types['repayment_count'] / repayment_types['repayment_count'].sum() * 100
    
    # Calculate customer repeat rate
    customer_loan_counts = loan_status.groupby('customer_id').size().reset_index(name='loan_count')
    repeat_customers = customer_loan_counts[customer_loan_counts['loan_count'] > 1]
    repeat_rate = len(repeat_customers) / len(customer_loan_counts) * 100
    
    print("Loan performance data prepared.")
    return loan_status, repayment_types, repeat_rate

# ===== STEP 1: RECOMMEND DATA-DRIVEN PRODUCT CHANGES =====
def recommend_product_changes(loan_status, repayment_types,repayments_df):
    print("\n=== STEP 1: RECOMMEND DATA-DRIVEN PRODUCT CHANGES ===")
    
    # 1.1 Optimal Fee Structure Adjustments
    print("\n1.1 Optimal Fee Structure Adjustments")
    
    # Analyze current fee structure
    fee_analysis = loan_status.groupby('loan_amount_bin').agg({
        'fee_percentage': ['mean', 'median', 'std'],
        'profit_margin': ['mean', 'median'],
        'is_defaulted': 'mean',
        'loan_amount': 'count'
    })
    
    fee_analysis.columns = ['avg_fee_pct', 'median_fee_pct', 'std_fee_pct', 'avg_profit_margin', 'median_profit_margin', 'default_rate', 'loan_count']
    fee_analysis = fee_analysis.reset_index()
    
    # Calculate default-adjusted profit margin
    fee_analysis['default_adjusted_profit'] = fee_analysis['avg_profit_margin'] * (1 - fee_analysis['default_rate'])
    
    # Identify optimal fee structure
    optimal_fees = []
    
    for _, row in fee_analysis.iterrows():
        if pd.isna(row['default_adjusted_profit']) or pd.isna(row['avg_fee_pct']):
            continue
            
        current_fee = row['avg_fee_pct']
        default_rate = row['default_rate']
        loan_bin = row['loan_amount_bin']
        
        # Recommend fee adjustments based on default rate and profit margin
        if default_rate > 0.05:  # High default rate
            recommended_fee = current_fee * 1.15  # Increase by 15%
            rationale = "Increase to offset higher default risk"
        elif default_rate < 0.01 and row['avg_profit_margin'] > 15:  # Low default rate and high profit
            recommended_fee = current_fee * 0.9  # Decrease by 10%
            rationale = "Decrease to improve competitiveness while maintaining profitability"
        else:
            recommended_fee = current_fee
            rationale = "Maintain current fee structure"
            
        optimal_fees.append({
            'loan_amount_bin': loan_bin,
            'current_fee_pct': current_fee,
            'recommended_fee_pct': recommended_fee,
            'default_rate': default_rate * 100,  # Convert to percentage
            'profit_margin': row['avg_profit_margin'],
            'rationale': rationale
        })
    
    optimal_fees_df = pd.DataFrame(optimal_fees)
    print("Recommended Fee Structure Adjustments:")
    if not optimal_fees_df.empty:
        print(optimal_fees_df[['loan_amount_bin', 'current_fee_pct', 'recommended_fee_pct', 'rationale']])
    else:
        print("Insufficient data to make fee structure recommendations.")
    
    # 1.2 Tenure Options Modifications
    print("\n1.2 Tenure Options Modifications")
    
    # Analyze performance by tenure
    tenure_analysis = loan_status.groupby('tenure_bin').agg({
        'profit_margin': ['mean', 'median'],
        'is_defaulted': 'mean',
        'loan_amount': ['count', 'mean'],
        'tenure_days': 'mean'
    })
    
    tenure_analysis.columns = ['avg_profit_margin', 'median_profit_margin', 'default_rate', 'loan_count', 'avg_loan_amount', 'avg_tenure_days']
    tenure_analysis = tenure_analysis.reset_index()
    
    # Identify current tenure options
    current_tenures = loan_status['tenure_days'].unique()
    current_tenures.sort()
    
    # Recommend new tenure options
    recommended_tenures = []
    
    # Check if we have limited tenure options
    if len(current_tenures) < 3:
        # Recommend expanding tenure options
        existing_max_tenure = max(current_tenures) if len(current_tenures) > 0 else 30
        
        # Suggest new tenure options based on common lending practices
        new_tenures = [existing_max_tenure]
        if existing_max_tenure < 30:
            new_tenures.append(30)
        if existing_max_tenure < 60:
            new_tenures.append(60)
        if existing_max_tenure < 90:
            new_tenures.append(90)
            
        for tenure in new_tenures:
            recommended_tenures.append({
                'current_tenure': "Not offered",
                'recommended_tenure': f"{tenure} days",
                'rationale': "Expand tenure options to meet diverse customer needs"
            })
    
    # Analyze existing tenures for potential modifications
    for tenure in current_tenures:
        tenure_data = loan_status[loan_status['tenure_days'] == tenure]
        default_rate = tenure_data['is_defaulted'].mean()
        profit_margin = tenure_data['profit_margin'].mean()
        
        if default_rate > 0.1:  # High default rate
            recommendation = "Consider removing or restructuring"
            rationale = "High default rate indicates potential risk"
        elif profit_margin < 5:  # Low profit margin
            recommendation = "Adjust fee structure or reconsider"
            rationale = "Low profitability"
        else:
            recommendation = "Maintain"
            rationale = "Good performance"
            
        recommended_tenures.append({
            'current_tenure': f"{tenure} days",
            'recommended_tenure': recommendation,
            'rationale': rationale
        })
    
    # Recommend new intermediate tenures if gaps exist
    if len(current_tenures) >= 2:
        for i in range(len(current_tenures) - 1):
            gap = current_tenures[i+1] - current_tenures[i]
            if gap > 30:  # Significant gap between tenure options
                new_tenure = current_tenures[i] + (gap // 2)
                recommended_tenures.append({
                    'current_tenure': "Not offered",
                    'recommended_tenure': f"{new_tenure} days",
                    'rationale': f"Fill gap between {current_tenures[i]} and {current_tenures[i+1]} day options"
                })
    
    tenure_recommendations_df = pd.DataFrame(recommended_tenures)
    print("Recommended Tenure Options Modifications:")
    print(tenure_recommendations_df)
    
    # 1.3 Loan Amount Limits
    print("\n1.3 Loan Amount Limits")
    
    # Analyze performance by loan amount
    amount_analysis = loan_status.groupby('loan_amount_bin').agg({
        'profit_margin': 'mean',
        'is_defaulted': 'mean',
        'loan_amount': ['count', 'mean', 'min', 'max']
    })
    
    amount_analysis.columns = ['avg_profit_margin', 'default_rate', 'loan_count', 'avg_loan_amount', 'min_loan_amount', 'max_loan_amount']
    amount_analysis = amount_analysis.reset_index()
    
    # Determine current loan amount limits
    current_min = loan_status['loan_amount'].min()
    current_max = loan_status['loan_amount'].max()
    
    # Recommend loan amount limits
    loan_limit_recommendations = []
    
    # Analyze each loan amount bin
    for _, row in amount_analysis.iterrows():
        if pd.isna(row['default_rate']) or pd.isna(row['avg_profit_margin']):
            continue
            
        loan_bin = row['loan_amount_bin']
        default_rate = row['default_rate']
        profit_margin = row['avg_profit_margin']
        
        # Make recommendations based on performance
        if default_rate > 0.05 and profit_margin < 10:
            recommendation = "Restrict or apply stricter criteria"
            rationale = "High default rate and low profitability"
        elif default_rate > 0.05:
            recommendation = "Increase fees or apply stricter criteria"
            rationale = "High default rate"
        elif profit_margin < 5:
            recommendation = "Reconsider or adjust fee structure"
            rationale = "Low profitability"
        elif profit_margin > 15 and default_rate < 0.02:
            recommendation = "Expand and promote"
            rationale = "High profitability and low default rate"
        else:
            recommendation = "Maintain current approach"
            rationale = "Balanced performance"
            
        loan_limit_recommendations.append({
            'loan_amount_bin': loan_bin,
            'default_rate': default_rate * 100,  # Convert to percentage
            'profit_margin': profit_margin,
            'recommendation': recommendation,
            'rationale': rationale
        })
    
    # Recommend overall loan amount limits
    best_performing_bin = None
    best_performance_score = -float('inf')
    
    for _, row in amount_analysis.iterrows():
        if pd.isna(row['default_rate']) or pd.isna(row['avg_profit_margin']):
            continue
            
        # Calculate a performance score (higher profit, lower default rate is better)
        performance_score = row['avg_profit_margin'] - (row['default_rate'] * 100)
        
        if performance_score > best_performance_score:
            best_performance_score = performance_score
            best_performing_bin = row['loan_amount_bin']
    
    # Recommend new min and max based on data
    recommended_min = current_min
    recommended_max = current_max
    
    # Adjust minimum if smallest loans are problematic
    smallest_bin = amount_analysis.iloc[0] if not amount_analysis.empty else None
    if smallest_bin is not None and not pd.isna(smallest_bin['default_rate']):
        if smallest_bin['default_rate'] > 0.05 or smallest_bin['avg_profit_margin'] < 5:
            # Recommend increasing the minimum
            recommended_min = min(smallest_bin['max_loan_amount'] * 1.2, smallest_bin['max_loan_amount'] + 100)
    
    # Adjust maximum if there's room for growth
    largest_bin = amount_analysis.iloc[-1] if not amount_analysis.empty else None
    if largest_bin is not None and not pd.isna(largest_bin['default_rate']):
        if largest_bin['default_rate'] < 0.03 and largest_bin['avg_profit_margin'] > 10:
            # Recommend increasing the maximum
            recommended_max = max(largest_bin['max_loan_amount'] * 1.2, largest_bin['max_loan_amount'] + 1000)
    
    # Add overall recommendations
    overall_recommendation = {
        'loan_amount_bin': 'Overall',
        'default_rate': loan_status['is_defaulted'].mean() * 100,
        'profit_margin': loan_status['profit_margin'].mean(),
        'recommendation': f"Set min: ${recommended_min:.0f}, max: ${recommended_max:.0f}",
        'rationale': f"Based on performance analysis, best performing segment: {best_performing_bin}"
    }
    
    loan_limit_recommendations.append(overall_recommendation)
    loan_limits_df = pd.DataFrame(loan_limit_recommendations)
    
    print("Recommended Loan Amount Limits:")
    print(loan_limits_df[['loan_amount_bin', 'recommendation', 'rationale']])
    
    # 1.4 Repayment Method Incentives
    print("\n1.4 Repayment Method Incentives")
    
    # Analyze repayment methods
    repayment_method_analysis = []
    
    # Merge repayment type with loan status
    repayment_with_loans = repayments_df.merge(loan_status[['customer_id', 'is_defaulted', 'loan_amount', 'expected_repayment']], 
                                              on='customer_id', how='left')
    
    # Analyze by repayment type
    repayment_performance = repayment_with_loans.groupby('repayment_type').agg({
        'repayment_amount': 'sum',
        'is_defaulted': 'mean',
        'customer_id': 'nunique'
    }).reset_index()
    
    repayment_performance.rename(columns={'customer_id': 'unique_customers'}, inplace=True)
    
    # Calculate on-time repayment rate by type (if data available)
    if 'repayment_date' in repayment_with_loans.columns and 'expected_repayment_date' in repayment_with_loans.columns:
        repayment_with_loans['days_late'] = (repayment_with_loans['repayment_date'] - repayment_with_loans['expected_repayment_date']).dt.days
        repayment_with_loans['is_on_time'] = repayment_with_loans['days_late'] <= 0
        
        on_time_by_type = repayment_with_loans.groupby('repayment_type')['is_on_time'].mean().reset_index()
        on_time_by_type.rename(columns={'is_on_time': 'on_time_rate'}, inplace=True)
        
        repayment_performance = repayment_performance.merge(on_time_by_type, on='repayment_type', how='left')
    
    # Make recommendations for each repayment type
    for _, row in repayment_performance.iterrows():
        repayment_type = row['repayment_type']
        default_rate = row['is_defaulted'] if not pd.isna(row['is_defaulted']) else 0
        on_time_rate = row.get('on_time_rate', None)
        
        # Determine if this method should be incentivized
        if on_time_rate is not None and on_time_rate > 0.9 and default_rate < 0.02:
            incentive = "Strong incentive recommended"
            rationale = "High on-time payment rate and low default rate"
        elif default_rate < 0.03:
            incentive = "Moderate incentive recommended"
            rationale = "Low default rate"
        else:
            incentive = "No incentive recommended"
            rationale = "Higher risk or insufficient performance advantage"
            
        # Suggest specific incentives
        if "Strong" in incentive or "Moderate" in incentive:
            if repayment_type == "Automatic":
                specific_incentive = "Fee discount of 5-10% for automatic repayment setup"
            else:
                specific_incentive = "Loyalty points or small discount for consistent manual repayments"
        else:
            specific_incentive = "N/A"
            
        repayment_method_analysis.append({
            'repayment_type': repayment_type,
            'default_rate': default_rate * 100 if not pd.isna(default_rate) else 0,
            'on_time_rate': on_time_rate * 100 if on_time_rate is not None else "N/A",
            'recommendation': incentive,
            'specific_incentive': specific_incentive,
            'rationale': rationale
        })
    
    repayment_incentives_df = pd.DataFrame(repayment_method_analysis)
    print("Recommended Repayment Method Incentives:")
    print(repayment_incentives_df[['repayment_type', 'recommendation', 'specific_incentive', 'rationale']])
    
    # Compile all recommendations
    product_recommendations = {
        'fee_structure': optimal_fees_df if not optimal_fees_df.empty else None,
        'tenure_options': tenure_recommendations_df,
        'loan_limits': loan_limits_df,
        'repayment_incentives': repayment_incentives_df
    }
    
    return product_recommendations

# ===== STEP 2: QUANTIFY POTENTIAL IMPACT OF RECOMMENDATIONS =====
def quantify_impact(loan_status, product_recommendations,repayments_df):
    print("\n=== STEP 2: QUANTIFY POTENTIAL IMPACT OF RECOMMENDATIONS ===")
    
    # 2.1 Expected Revenue Increase
    print("\n2.1 Expected Revenue Increase")
    
    # Calculate baseline metrics
    total_loans = len(loan_status)
    total_loan_amount = loan_status['loan_amount'].sum()
    total_fee_revenue = loan_status['loan_fee'].sum()
    total_profit = loan_status['profit'].sum()
    average_fee_percentage = (total_fee_revenue / total_loan_amount) * 100
    
    print(f"Baseline Metrics:")
    print(f"- Total Loans: {total_loans}")
    print(f"- Total Loan Amount: ${total_loan_amount:,.2f}")
    print(f"- Total Fee Revenue: ${total_fee_revenue:,.2f}")
    print(f"- Total Profit: ${total_profit:,.2f}")
    print(f"- Average Fee Percentage: {average_fee_percentage:.2f}%")
    
    # Estimate impact of fee structure changes
    fee_impact = 0
    if product_recommendations['fee_structure'] is not None and not product_recommendations['fee_structure'].empty:
        fee_df = product_recommendations['fee_structure']
        
        for _, row in fee_df.iterrows():
            loan_bin = row['loan_amount_bin']
            current_fee = row['current_fee_pct']
            recommended_fee = row['recommended_fee_pct']
            
            # Get loans in this bin
            bin_loans = loan_status[loan_status['loan_amount_bin'] == loan_bin]
            bin_loan_amount = bin_loans['loan_amount'].sum()
            
            # Calculate fee impact
            fee_change_pct = (recommended_fee - current_fee) / current_fee if current_fee > 0 else 0
            bin_impact = bin_loan_amount * (fee_change_pct / 100) * current_fee
            fee_impact += bin_impact
    
    # Estimate impact of tenure options
    tenure_impact = 0
    new_tenures_count = 0
    
    for _, row in product_recommendations['tenure_options'].iterrows():
        if "Not offered" in row['current_tenure'] and "days" in row['recommended_tenure']:
            new_tenures_count += 1
    
    # Assume each new tenure option increases loan volume by 2-5%
    if new_tenures_count > 0:
        volume_increase_pct = min(new_tenures_count * 3, 15)  # Cap at 15%
        tenure_impact = (total_fee_revenue * volume_increase_pct / 100)
    
    # Estimate impact of loan limit changes
    loan_limit_impact = 0
    
    # Check if we're recommending increasing the maximum loan amount
    overall_rec = product_recommendations['loan_limits'][product_recommendations['loan_limits']['loan_amount_bin'] == 'Overall']
    if not overall_rec.empty:
        rec_text = overall_rec.iloc[0]['recommendation']
        if "max:" in rec_text:
            # Extract recommended max
            try:
                current_max = loan_status['loan_amount'].max()
                recommended_max = float(rec_text.split("max: $")[1].split(",")[0].replace(",", ""))
                
                if recommended_max > current_max:
                    # Estimate impact of higher loan limits
                    max_increase_pct = (recommended_max - current_max) / current_max
                    # Assume 10% of customers would take larger loans
                    loan_limit_impact = total_fee_revenue * max_increase_pct * 0.1
            except:
                pass
    
    # Estimate impact of repayment incentives
    repayment_impact = 0
    
    for _, row in product_recommendations['repayment_incentives'].iterrows():
        if "Strong" in row['recommendation'] or "Moderate" in row['recommendation']:
            repayment_type = row['repayment_type']
            
            # Estimate impact based on repayment type
            if repayment_type == "Automatic":
                # Assume incentives increase automatic payments by 15%
                # But with a 5-10% discount on fees
                auto_payment_increase = 0.15
                fee_discount = 0.075  # Average of 5-10%
                
                # Calculate net impact: more volume but lower fees per loan
                current_auto_loans = loan_status[loan_status['customer_id'].isin(
                    repayments_df[repayments_df['repayment_type'] == 'Automatic']['customer_id']
                )]
                
                auto_loan_revenue = current_auto_loans['loan_fee'].sum()
                auto_loan_count = len(current_auto_loans)
                
                # New revenue: more loans but lower fee per loan
                new_auto_loan_count = auto_loan_count * (1 + auto_payment_increase)
                new_auto_loan_revenue = (new_auto_loan_count / auto_loan_count) * auto_loan_revenue * (1 - fee_discount)
                
                auto_impact = new_auto_loan_revenue - auto_loan_revenue
                repayment_impact += auto_impact
            else:
                # For manual payments, assume smaller impact
                manual_payment_increase = 0.05
                fee_discount = 0.03
                
                current_manual_loans = loan_status[loan_status['customer_id'].isin(
                    repayments_df[repayments_df['repayment_type'] == 'Manual']['customer_id']
                )]
                
                manual_loan_revenue = current_manual_loans['loan_fee'].sum()
                manual_loan_count = len(current_manual_loans)
                
                new_manual_loan_count = manual_loan_count * (1 + manual_payment_increase)
                new_manual_loan_revenue = (new_manual_loan_count / manual_loan_count) * manual_loan_revenue * (1 - fee_discount)
                
                manual_impact = new_manual_loan_revenue - manual_loan_revenue
                repayment_impact += manual_impact
    
    # Calculate total expected revenue increase
    total_revenue_impact = fee_impact + tenure_impact + loan_limit_impact + repayment_impact
    revenue_increase_pct = (total_revenue_impact / total_fee_revenue) * 100 if total_fee_revenue > 0 else 0
    
    print(f"\nExpected Revenue Impact:")
    print(f"- Fee Structure Changes: ${fee_impact:,.2f}")
    print(f"- New Tenure Options: ${tenure_impact:,.2f}")
    print(f"- Loan Limit Adjustments: ${loan_limit_impact:,.2f}")
    print(f"- Repayment Incentives: ${repayment_impact:,.2f}")
    print(f"- Total Expected Revenue Increase: ${total_revenue_impact:,.2f} ({revenue_increase_pct:.2f}%)")
    
    # 2.2 Risk Reduction Potential
    print("\n2.2 Risk Reduction Potential")
    
    # Calculate baseline risk metrics
    total_defaults = loan_status['is_defaulted'].sum()
    default_rate = (total_defaults / total_loans) * 100
    default_amount = loan_status[loan_status['is_defaulted']]['outstanding_amount'].sum()
    
    print(f"Baseline Risk Metrics:")
    print(f"- Total Defaults: {total_defaults}")
    print(f"- Default Rate: {default_rate:.2f}%")
    print(f"- Default Amount: ${default_amount:,.2f}")
    
    # Estimate impact of fee structure changes on risk
    fee_risk_impact = 0
    if product_recommendations['fee_structure'] is not None and not product_recommendations['fee_structure'].empty:
        for _, row in product_recommendations['fee_structure'].iterrows():
            if "Increase" in row['rationale'] and "offset higher default risk" in row['rationale']:
                loan_bin = row['loan_amount_bin']
                bin_defaults = loan_status[(loan_status['loan_amount_bin'] == loan_bin) & loan_status['is_defaulted']]
                bin_default_amount = bin_defaults['outstanding_amount'].sum()
                
                # Assume higher fees reduce volume in high-risk segments by 10%
                fee_risk_impact += bin_default_amount * 0.1
    
    # Estimate impact of loan limit changes on risk
    loan_limit_risk_impact = 0
    
    for _, row in product_recommendations['loan_limits'].iterrows():
        if "Restrict" in row['recommendation'] or "stricter criteria" in row['recommendation']:
            loan_bin = row['loan_amount_bin']
            if loan_bin != 'Overall':
                bin_defaults = loan_status[(loan_status['loan_amount_bin'] == loan_bin) & loan_status['is_defaulted']]
                bin_default_amount = bin_defaults['outstanding_amount'].sum()
                
                # Assume restrictions reduce defaults in high-risk segments by 20%
                loan_limit_risk_impact += bin_default_amount * 0.2
    
    # Estimate impact of repayment incentives on risk
    repayment_risk_impact = 0
    
    for _, row in product_recommendations['repayment_incentives'].iterrows():
        if "Strong" in row['recommendation'] and "Automatic" in row['repayment_type']:
            # Automatic payments typically reduce default rates
            # Assume 20% reduction in defaults for customers switching to automatic payments
            auto_payment_increase = 0.15  # From revenue calculation
            
            # Calculate potential default reduction
            manual_defaults = loan_status[loan_status['customer_id'].isin(
                repayments_df[repayments_df['repayment_type'] == 'Manual']['customer_id']
            ) & loan_status['is_defaulted']]
            
            manual_default_amount = manual_defaults['outstanding_amount'].sum()
            
            # Estimate defaults prevented by switching to automatic
            repayment_risk_impact += manual_default_amount * auto_payment_increase * 0.2
    
    # Calculate total risk reduction
    total_risk_reduction = fee_risk_impact + loan_limit_risk_impact + repayment_risk_impact
    risk_reduction_pct = (total_risk_reduction / default_amount) * 100 if default_amount > 0 else 0
    
    print(f"\nExpected Risk Reduction:")
    print(f"- Fee Structure Changes: ${fee_risk_impact:,.2f}")
    print(f"- Loan Limit Adjustments: ${loan_limit_risk_impact:,.2f}")
    print(f"- Repayment Incentives: ${repayment_risk_impact:,.2f}")
    print(f"- Total Expected Risk Reduction: ${total_risk_reduction:,.2f} ({risk_reduction_pct:.2f}%)")
    
    # 2.3 Customer Experience Improvements
    print("\n2.3 Customer Experience Improvements")
    
    # Calculate baseline customer metrics
    unique_customers = loan_status['customer_id'].nunique()
    repeat_rate = (total_loans - unique_customers) / unique_customers * 100 if unique_customers > 0 else 0
    
    print(f"Baseline Customer Metrics:")
    print(f"- Unique Customers: {unique_customers}")
    print(f"- Repeat Rate: {repeat_rate:.2f}%")
    
    # Estimate impact on customer experience
    
    # Impact of new tenure options on customer satisfaction
    tenure_satisfaction_impact = 0
    if new_tenures_count > 0:
        # Assume each new tenure option improves satisfaction for 5% of customers
        tenure_satisfaction_impact = min(new_tenures_count * 5, 20)  # Cap at 20%
    
    # Impact of loan limit changes on customer satisfaction
    loan_limit_satisfaction_impact = 0
    
    # Check if we're recommending increasing the maximum loan amount
    if not overall_rec.empty:
        rec_text = overall_rec.iloc[0]['recommendation']
        if "max:" in rec_text:
            # Assume higher loan limits improve satisfaction for 10% of customers
            loan_limit_satisfaction_impact = 10
    
    # Impact of repayment incentives on customer satisfaction
    repayment_satisfaction_impact = 0
    
    for _, row in product_recommendations['repayment_incentives'].iterrows():
        if "Strong" in row['recommendation'] or "Moderate" in row['recommendation']:
            # Assume incentives improve satisfaction for 15% of customers
            repayment_satisfaction_impact = 15
            break
    
    # Estimate impact on repeat rate
    current_repeat_rate = repeat_rate
    
    # Assume each percentage point of satisfaction improvement increases repeat rate by 0.5%
    total_satisfaction_improvement = tenure_satisfaction_impact + loan_limit_satisfaction_impact + repayment_satisfaction_impact
    repeat_rate_increase = total_satisfaction_improvement * 0.5
    
    new_repeat_rate = current_repeat_rate + repeat_rate_increase
    
    # Calculate revenue impact of improved repeat rate
    avg_profit_per_customer = total_profit / unique_customers if unique_customers > 0 else 0
    additional_repeats = unique_customers * (repeat_rate_increase / 100)
    repeat_revenue_impact = additional_repeats * avg_profit_per_customer
    
    print(f"\nExpected Customer Experience Improvements:")
    print(f"- Tenure Options Impact: {tenure_satisfaction_impact:.2f}% satisfaction improvement")
    print(f"- Loan Limit Impact: {loan_limit_satisfaction_impact:.2f}% satisfaction improvement")
    print(f"- Repayment Incentives Impact: {repayment_satisfaction_impact:.2f}% satisfaction improvement")
    print(f"- Total Satisfaction Improvement: {total_satisfaction_improvement:.2f}%")
    print(f"- Current Repeat Rate: {current_repeat_rate:.2f}%")
    print(f"- Projected New Repeat Rate: {new_repeat_rate:.2f}%")
    print(f"- Additional Revenue from Improved Repeat Rate: ${repeat_revenue_impact:,.2f}")
    
    # Compile impact metrics
    impact_metrics = {
        'revenue_impact': {
            'fee_impact': fee_impact,
            'tenure_impact': tenure_impact,
            'loan_limit_impact': loan_limit_impact,
            'repayment_impact': repayment_impact,
            'total_impact': total_revenue_impact,
            'impact_percentage': revenue_increase_pct
        },
        'risk_reduction': {
            'fee_risk_impact': fee_risk_impact,
            'loan_limit_risk_impact': loan_limit_risk_impact,
            'repayment_risk_impact': repayment_risk_impact,
            'total_risk_reduction': total_risk_reduction,
            'risk_reduction_percentage': risk_reduction_pct
        },
        'customer_experience': {
            'tenure_satisfaction_impact': tenure_satisfaction_impact,
            'loan_limit_satisfaction_impact': loan_limit_satisfaction_impact,
            'repayment_satisfaction_impact': repayment_satisfaction_impact,
            'total_satisfaction_improvement': total_satisfaction_improvement,
            'current_repeat_rate': current_repeat_rate,
            'projected_repeat_rate': new_repeat_rate,
            'repeat_revenue_impact': repeat_revenue_impact
        }
    }
    
    return impact_metrics

# ===== STEP 3: COMPILE COMPREHENSIVE FINAL REPORT =====
def compile_final_report(loan_status, product_recommendations, impact_metrics):
    print("\n=== STEP 3: COMPILE COMPREHENSIVE FINAL REPORT ===")
    
    # Create report content
    report = """# Loan Product Enhancement Recommendations
## Executive Summary

Based on comprehensive analysis of loan performance data, we recommend several strategic product enhancements to optimize revenue, reduce risk, and improve customer experience. Our key recommendations include:

1. **Fee Structure Adjustments**: Implement targeted fee changes across loan amount segments to balance profitability and competitiveness.
2. **Tenure Options Expansion**: Introduce new tenure options to meet diverse customer needs and fill gaps in current offerings.
3. **Loan Amount Limit Refinements**: Adjust minimum and maximum loan amounts based on performance data to optimize risk-reward balance.
4. **Repayment Method Incentives**: Offer incentives for automatic repayments to reduce default rates and improve operational efficiency.

These recommendations are projected to deliver:
- Revenue increase of ${:,.2f} ({:.2f}%)
- Risk reduction of ${:,.2f} ({:.2f}%)
- Customer satisfaction improvement of {:.2f}%, potentially increasing repeat business by {:.2f}%

## Methodology and Assumptions

This analysis was conducted using loan disbursement and repayment data covering {:,} loans to {:,} unique customers. Our methodology included:

1. **Data Preparation**: Cleaning and structuring loan data to enable comprehensive analysis.
2. **Performance Analysis**: Evaluating loan performance across different segments by loan amount, tenure, and repayment method.
3. **Profitability Assessment**: Calculating profit margins and default rates to identify optimal product configurations.
4. **Impact Modeling**: Estimating the financial and customer impact of proposed changes using conservative assumptions.

Key assumptions in our impact calculations include:
- New tenure options will increase loan volume by approximately 3% per option
- Fee adjustments will affect customer behavior in high-risk segments
- Automatic payment incentives will increase adoption by 15%
- Customer satisfaction improvements translate to repeat business at a 0.5:1 ratio

## Key Findings

""".format(
        impact_metrics['revenue_impact']['total_impact'],
        impact_metrics['revenue_impact']['impact_percentage'],
        impact_metrics['risk_reduction']['total_risk_reduction'],
        impact_metrics['risk_reduction']['risk_reduction_percentage'],
        impact_metrics['customer_experience']['total_satisfaction_improvement'],
        impact_metrics['customer_experience']['projected_repeat_rate'] - impact_metrics['customer_experience']['current_repeat_rate'],
        len(loan_status),
        loan_status['customer_id'].nunique()
    )
    
    # Add fee structure findings
    report += """### Fee Structure Analysis

| Loan Amount Range | Current Fee % | Recommended Fee % | Default Rate | Profit Margin | Rationale |
|-------------------|--------------|-------------------|--------------|---------------|-----------|
"""
    
    if product_recommendations['fee_structure'] is not None and not product_recommendations['fee_structure'].empty:
        for _, row in product_recommendations['fee_structure'].iterrows():
            report += "| {} | {:.2f}% | {:.2f}% | {:.2f}% | {:.2f}% | {} |\n".format(
                row['loan_amount_bin'],
                row['current_fee_pct'],
                row['recommended_fee_pct'],
                row['default_rate'],
                row['profit_margin'],
                row['rationale']
            )
    else:
        report += "| No data available | - | - | - | - | - |\n"
    
    # Add tenure options findings
    report += """\n### Tenure Options Analysis

| Current Tenure | Recommendation | Rationale |
|----------------|----------------|-----------|
"""
    
    for _, row in product_recommendations['tenure_options'].iterrows():
        report += "| {} | {} | {} |\n".format(
            row['current_tenure'],
            row['recommended_tenure'],
            row['rationale']
        )
    
    # Add loan amount findings
    report += """\n### Loan Amount Limits Analysis

| Loan Amount Range | Default Rate | Profit Margin | Recommendation | Rationale |
|-------------------|--------------|---------------|----------------|-----------|
"""
    
    for _, row in product_recommendations['loan_limits'].iterrows():
        report += "| {} | {:.2f}% | {:.2f}% | {} | {} |\n".format(
            row['loan_amount_bin'],
            row['default_rate'] if not pd.isna(row['default_rate']) else 0,
            row['profit_margin'] if not pd.isna(row['profit_margin']) else 0,
            row['recommendation'],
            row['rationale']
        )
    
    # Add repayment method findings
    report += """\n### Repayment Method Analysis

| Repayment Type | Default Rate | On-Time Rate | Recommendation | Specific Incentive | Rationale |
|----------------|--------------|--------------|----------------|-------------------|-----------|
"""
    
    for _, row in product_recommendations['repayment_incentives'].iterrows():
        report += "| {} | {:.2f}% | {} | {} | {} | {} |\n".format(
            row['repayment_type'],
            row['default_rate'],
            row['on_time_rate'] if not isinstance(row['on_time_rate'], str) else row['on_time_rate'],
            row['recommendation'],
            row['specific_incentive'],
            row['rationale']
        )
    
    # Add detailed impact analysis
    report += """\n## Quantified Impact Analysis

### Revenue Impact

- **Fee Structure Changes**: ${:,.2f}
- **New Tenure Options**: ${:,.2f}
- **Loan Limit Adjustments**: ${:,.2f}
- **Repayment Incentives**: ${:,.2f}
- **Total Expected Revenue Increase**: ${:,.2f} ({:.2f}%)

### Risk Reduction

- **Fee Structure Changes**: ${:,.2f}
- **Loan Limit Adjustments**: ${:,.2f}
- **Repayment Incentives**: ${:,.2f}
- **Total Expected Risk Reduction**: ${:,.2f} ({:.2f}%)

### Customer Experience Improvements

- **Tenure Options Impact**: {:.2f}% satisfaction improvement
- **Loan Limit Impact**: {:.2f}% satisfaction improvement
- **Repayment Incentives Impact**: {:.2f}% satisfaction improvement
- **Total Satisfaction Improvement**: {:.2f}%
- **Current Repeat Rate**: {:.2f}%
- **Projected New Repeat Rate**: {:.2f}%
- **Additional Revenue from Improved Repeat Rate**: ${:,.2f}

""".format(
        impact_metrics['revenue_impact']['fee_impact'],
        impact_metrics['revenue_impact']['tenure_impact'],
        impact_metrics['revenue_impact']['loan_limit_impact'],
        impact_metrics['revenue_impact']['repayment_impact'],
        impact_metrics['revenue_impact']['total_impact'],
        impact_metrics['revenue_impact']['impact_percentage'],
        impact_metrics['risk_reduction']['fee_risk_impact'],
        impact_metrics['risk_reduction']['loan_limit_risk_impact'],
        impact_metrics['risk_reduction']['repayment_risk_impact'],
        impact_metrics['risk_reduction']['total_risk_reduction'],
        impact_metrics['risk_reduction']['risk_reduction_percentage'],
        impact_metrics['customer_experience']['tenure_satisfaction_impact'],
        impact_metrics['customer_experience']['loan_limit_satisfaction_impact'],
        impact_metrics['customer_experience']['repayment_satisfaction_impact'],
        impact_metrics['customer_experience']['total_satisfaction_improvement'],
        impact_metrics['customer_experience']['current_repeat_rate'],
        impact_metrics['customer_experience']['projected_repeat_rate'],
        impact_metrics['customer_experience']['repeat_revenue_impact']
    )
    
    # Add recommendations and implementation roadmap
    report += """## Recommendations and Implementation Roadmap

Based on our analysis, we recommend the following product enhancements:

### 1. Fee Structure Adjustments

**Recommendation:**
"""
    
    if product_recommendations['fee_structure'] is not None and not product_recommendations['fee_structure'].empty:
        for _, row in product_recommendations['fee_structure'].iterrows():
            if abs(row['recommended_fee_pct'] - row['current_fee_pct']) > 0.1:
                report += "- {} loans: Adjust fee from {:.2f}% to {:.2f}% ({})\n".format(
                    row['loan_amount_bin'],
                    row['current_fee_pct'],
                    row['recommended_fee_pct'],
                    "increase" if row['recommended_fee_pct'] > row['current_fee_pct'] else "decrease"
                )
    else:
        report += "- Maintain current fee structure pending further data collection\n"
    
    report += """
**Implementation Timeline:**
- Month 1: Finalize fee structure changes and update systems
- Month 2: Implement new fee structure
- Month 3: Monitor impact and make adjustments as needed

### 2. Tenure Options Expansion

**Recommendation:**
"""
    
    new_tenures = []
    for _, row in product_recommendations['tenure_options'].iterrows():
        if "Not offered" in row['current_tenure'] and "days" in row['recommended_tenure']:
            tenure_days = row['recommended_tenure'].split(" ")[0]
            new_tenures.append(tenure_days)
    
    if new_tenures:
        for tenure in new_tenures:
            report += f"- Introduce new {tenure} day tenure option\n"
    else:
        report += "- Maintain current tenure options pending further data collection\n"
    
    report += """
**Implementation Timeline:**
- Month 1: Update loan management system to support new tenure options
- Month 2: Launch new tenure options with marketing campaign
- Month 3-4: Monitor adoption and performance of new options

### 3. Loan Amount Limit Refinements

**Recommendation:**
"""
    
    overall_rec = product_recommendations['loan_limits'][product_recommendations['loan_limits']['loan_amount_bin'] == 'Overall']
    if not overall_rec.empty:
        report += "- " + overall_rec.iloc[0]['recommendation'] + "\n"
    
    for _, row in product_recommendations['loan_limits'].iterrows():
        if row['loan_amount_bin'] != 'Overall' and "Restrict" in row['recommendation']:
            report += f"- {row['loan_amount_bin']} loans: {row['recommendation']}\n"
    
    report += """
**Implementation Timeline:**
- Month 1: Update loan amount limits in systems
- Month 1-2: Train customer service team on new limits and criteria
- Month 2-3: Monitor impact on application volume and approval rates

### 4. Repayment Method Incentives

**Recommendation:**
"""
    
    for _, row in product_recommendations['repayment_incentives'].iterrows():
        if "Strong" in row['recommendation'] or "Moderate" in row['recommendation']:
            report += f"- {row['repayment_type']} payments: {row['specific_incentive']}\n"
    
    report += """
**Implementation Timeline:**
- Month 1: Design incentive program details and update systems
- Month 2: Launch incentive program with targeted customer communications
- Month 3-6: Monitor adoption rates and impact on repayment behavior

## Conclusion

The recommended product enhancements represent a balanced approach to improving profitability while managing risk and enhancing customer experience. By implementing these changes, we project a significant positive impact on both the top and bottom line, with minimal disruption to existing operations.

We recommend a phased implementation approach, starting with the fee structure adjustments and repayment incentives, followed by tenure options and loan amount limit changes. This approach will allow for careful monitoring and adjustment at each stage.

Regular review of performance metrics will be essential to validate the impact of these changes and identify opportunities for further optimization.
"""
    
    # Save the report to a file
    with open("product_enhancement_recommendations.md", "w") as f:
        f.write(report)
    
    print("Comprehensive final report generated and saved as 'product_enhancement_recommendations.md'")
    return report

# ===== STEP 4: PREPARE PRESENTATION-READY VISUALIZATIONS =====
def prepare_visualizations(loan_status, product_recommendations, impact_metrics):
    print("\n=== STEP 4: PREPARE PRESENTATION-READY VISUALIZATIONS ===")
    
    # 4.1 Fee Structure Visualization
    print("\n4.1 Fee Structure Visualization")
    
    if product_recommendations['fee_structure'] is not None and not product_recommendations['fee_structure'].empty:
        fee_df = product_recommendations['fee_structure'].copy()
        
        # Create a bar chart comparing current and recommended fees
        fig1 = go.Figure()
        
        fig1.add_trace(go.Bar(
            x=fee_df['loan_amount_bin'],
            y=fee_df['current_fee_pct'],
            name='Current Fee %',
            marker_color='lightblue'
        ))
        
        fig1.add_trace(go.Bar(
            x=fee_df['loan_amount_bin'],
            y=fee_df['recommended_fee_pct'],
            name='Recommended Fee %',
            marker_color='darkblue'
        ))
        
        fig1.update_layout(
            title='Current vs. Recommended Fee Structure',
            xaxis_title='Loan Amount Range',
            yaxis_title='Fee Percentage',
            barmode='group',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        fig1.write_html("fee_structure_comparison.html")
        fig1.write_image("fee_structure_comparison.png")
    
    # 4.2 Tenure Options Visualization
    print("\n4.2 Tenure Options Visualization")
    
    # Extract current and recommended tenures
    current_tenures = []
    recommended_tenures = []
    
    for _, row in product_recommendations['tenure_options'].iterrows():
        if "days" in row['current_tenure']:
            try:
                days = int(row['current_tenure'].split(" ")[0])
                current_tenures.append(days)
            except:
                pass
        
        if "days" in row['recommended_tenure'] and "Not offered" in row['current_tenure']:
            try:
                days = int(row['recommended_tenure'].split(" ")[0])
                recommended_tenures.append(days)
            except:
                pass
    
    # Combine all tenures for visualization
    all_tenures = sorted(list(set(current_tenures + recommended_tenures)))
    
    # Create data for visualization
    tenure_data = []
    
    for tenure in all_tenures:
        status = "Current" if tenure in current_tenures else "New (Recommended)"
        tenure_data.append({
            'tenure_days': tenure,
            'status': status
        })
    
    tenure_df = pd.DataFrame(tenure_data)
    
    # Create a timeline-style visualization
    fig2 = px.scatter(
        tenure_df,
        x='tenure_days',
        y='status',
        color='status',
        size=[20] * len(tenure_df),  # Consistent size
        labels={'tenure_days': 'Tenure (Days)', 'status': 'Status'},
        title='Current and Recommended Tenure Options',
        color_discrete_map={'Current': 'blue', 'New (Recommended)': 'green'}
    )
    
    fig2.update_layout(
        xaxis=dict(
            tickmode='array',
            tickvals=all_tenures,
            ticktext=[f"{t} days" for t in all_tenures]
        ),
        yaxis=dict(
            categoryorder='array',
            categoryarray=['New (Recommended)', 'Current']
        ),
        showlegend=True
    )
    
    # Add vertical lines to show the timeline
    for tenure in all_tenures:
        fig2.add_shape(
            type="line",
            x0=tenure,
            y0=-0.5,
            x1=tenure,
            y1=1.5,
            line=dict(color="lightgray", width=1, dash="dot")
        )
    
    fig2.write_html("tenure_options_visualization.html")
    fig2.write_image("tenure_options_visualization.png")
    
    # 4.3 Loan Amount Limits Visualization
    print("\n4.3 Loan Amount Limits Visualization")
    
    # Extract data for visualization
    loan_limits_df = product_recommendations['loan_limits'].copy()
    
    # Filter out the 'Overall' row for this visualization
    loan_limits_df = loan_limits_df[loan_limits_df['loan_amount_bin'] != 'Overall']
    
    # Create a scatter plot of default rate vs. profit margin
    fig3 = px.scatter(
        loan_limits_df,
        x='default_rate',
        y='profit_margin',
        color='recommendation',
        size=[30] * len(loan_limits_df),  # Consistent size
        text='loan_amount_bin',
        labels={
            'default_rate': 'Default Rate (%)',
            'profit_margin': 'Profit Margin (%)',
            'recommendation': 'Recommendation'
        },
        title='Loan Amount Segments: Risk vs. Profitability'
    )
    
    fig3.update_traces(
        textposition='top center',
        marker=dict(line=dict(width=1, color='DarkSlateGrey'))
    )
    
    fig3.update_layout(
        xaxis=dict(range=[-1, max(loan_limits_df['default_rate']) * 1.1]),
        yaxis=dict(range=[-1, max(loan_limits_df['profit_margin']) * 1.1]),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # Add quadrant lines
    fig3.add_shape(
        type="line",
        x0=0,
        y0=10,
        x1=max(loan_limits_df['default_rate']) * 1.1,
        y1=10,
        line=dict(color="gray", width=1, dash="dash")
    )
    
    fig3.add_shape(
        type="line",
        x0=3,
        y0=0,
        x1=3,
        y1=max(loan_limits_df['profit_margin']) * 1.1,
        line=dict(color="gray", width=1, dash="dash")
    )
    
    # Add quadrant labels
    fig3.add_annotation(
        x=1.5,
        y=max(loan_limits_df['profit_margin']) * 0.8,
        text="Low Risk, High Profit<br>(Expand)",
        showarrow=False,
        font=dict(size=10, color="green")
    )
    
    fig3.add_annotation(
        x=max(loan_limits_df['default_rate']) * 0.8,
        y=max(loan_limits_df['profit_margin']) * 0.8,
        text="High Risk, High Profit<br>(Adjust Fees)",
        showarrow=False,
        font=dict(size=10, color="orange")
    )
    
    fig3.add_annotation(
        x=1.5,
        y=5,
        text="Low Risk, Low Profit<br>(Optimize)",
        showarrow=False,
        font=dict(size=10, color="blue")
    )
    
    fig3.add_annotation(
        x=max(loan_limits_df['default_rate']) * 0.8,
        y=5,
        text="High Risk, Low Profit<br>(Restrict)",
        showarrow=False,
        font=dict(size=10, color="red")
    )
    
    fig3.write_html("loan_amount_risk_profitability.html")
    fig3.write_image("loan_amount_risk_profitability.png")
    
# 4.4 Repayment Method Visualization
    print("\n4.4 Repayment Method Visualization")

    # Extract data for visualization
    repayment_df = product_recommendations['repayment_incentives'].copy()

    # Create a grouped bar chart for repayment methods
    fig4 = go.Figure()

    # Add default rate bars
    fig4.add_trace(go.Bar(
        x=repayment_df['repayment_type'],
        y=repayment_df['default_rate'],
        name='Default Rate (%)',
        marker_color='red'
    ))

    # Add on-time rate bars if available
    if 'on_time_rate' in repayment_df.columns and not all(isinstance(x, str) for x in repayment_df['on_time_rate']):
        on_time_values = []
        for val in repayment_df['on_time_rate']:
            if isinstance(val, str) and val == 'N/A':
                on_time_values.append(0)
            else:
                on_time_values.append(val)

        fig4.add_trace(go.Bar(
            x=repayment_df['repayment_type'],
            y=on_time_values,
            name='On-Time Rate (%)',
            marker_color='green'
        ))

    # Add recommendation indicators
    recommendations = []
    for _, row in repayment_df.iterrows():
        if "Strong" in row['recommendation']:
            recommendations.append("Strong Incentive")
        elif "Moderate" in row['recommendation']:
            recommendations.append("Moderate Incentive")
        else:
            recommendations.append("No Incentive")

    # Add text annotations for recommendations
    for i, rec in enumerate(recommendations):
        color = "green" if "Strong" in rec else "orange" if "Moderate" in rec else "red"
        fig4.add_annotation(
            x=repayment_df['repayment_type'].iloc[i],
            y=repayment_df['default_rate'].iloc[i] + 5,  # Position above the bar
            text=rec,
            showarrow=False,
            font=dict(size=12, color=color)
        )

    fig4.update_layout(
        title='Repayment Methods: Performance and Recommendations',
        xaxis_title='Repayment Type',
        yaxis_title='Rate (%)',
        barmode='group',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig4.write_html("repayment_methods_visualization.html")
    fig4.write_image("repayment_methods_visualization.png")

    # 4.5 Impact Summary Visualization
    print("\n4.5 Impact Summary Visualization")

    # Create data for impact visualization
    impact_data = [
        {'category': 'Revenue Increase', 'value': impact_metrics['revenue_impact']['total_impact']},
        {'category': 'Risk Reduction', 'value': impact_metrics['risk_reduction']['total_risk_reduction']},
        {'category': 'Customer Experience', 'value': impact_metrics['customer_experience']['repeat_revenue_impact']}
    ]

    impact_df = pd.DataFrame(impact_data)

    # Create a bar chart for impact summary
    fig5 = px.bar(
        impact_df,
        x='category',
        y='value',
        color='category',
        text_auto='.2s',
        labels={'category': 'Impact Category', 'value': 'Financial Impact ($)'},
        title='Projected Financial Impact of Recommendations'
    )

    fig5.update_traces(textposition='outside')

    fig5.update_layout(
        showlegend=False,
        xaxis=dict(categoryorder='total descending')
    )

    fig5.write_html("impact_summary_visualization.html")
    fig5.write_image("impact_summary_visualization.png")

    # 4.6 Implementation Roadmap Visualization
    print("\n4.6 Implementation Roadmap Visualization")

    # Create data for implementation roadmap
    roadmap_data = [
        {'task': 'Fee Structure Adjustments', 'start': 0, 'duration': 3, 'description': 'Update fee structure'},
        {'task': 'Tenure Options Expansion', 'start': 1, 'duration': 4, 'description': 'Add new tenure options'},
        {'task': 'Loan Amount Limit Refinements', 'start': 0, 'duration': 3, 'description': 'Update loan limits'},
        {'task': 'Repayment Method Incentives', 'start': 1, 'duration': 6, 'description': 'Implement incentives'}
    ]

    roadmap_df = pd.DataFrame(roadmap_data)

    # Calculate end month for each task
    roadmap_df['end'] = roadmap_df['start'] + roadmap_df['duration']

    # Create a Gantt chart for implementation roadmap
    fig6 = px.timeline(
        roadmap_df,
        x_start='start',
        x_end='end',
        y='task',
        color='task',
        text='description',
        labels={'task': 'Initiative', 'start': 'Month', 'end': 'Month'},
        title='Implementation Roadmap'
    )

    fig6.update_yaxes(autorange="reversed")

    fig6.update_xaxes(
        tickvals=list(range(7)),
        ticktext=['Month ' + str(i+1) for i in range(7)]
    )

    fig6.update_layout(
        showlegend=False,
        xaxis_title="Timeline",
        yaxis_title="Initiative"
    )

    fig6.write_html("implementation_roadmap.html")
    fig6.write_image("implementation_roadmap.png")

    print("Presentation-ready visualizations created and saved.")

    return {
        'fee_structure': 'fee_structure_comparison.png',
        'tenure_options': 'tenure_options_visualization.png',
        'loan_amount_limits': 'loan_amount_risk_profitability.png',
        'repayment_methods': 'repayment_methods_visualization.png',
        'impact_summary': 'impact_summary_visualization.png',
        'implementation_roadmap': 'implementation_roadmap.png'
    }

# ===== STEP 5: DOCUMENT SQL QUERIES AND PYTHON CODE =====
def document_code():
    print("\n=== STEP 5: DOCUMENT SQL QUERIES AND PYTHON CODE ===")

    # Create documentation content
    documentation = """# Code Documentation: Product Enhancement Analysis

## Overview

This document provides documentation for the SQL queries and Python code used in the loan product enhancement analysis. The analysis was conducted to identify opportunities for optimizing the loan product offering based on historical performance data.

## Data Sources

The analysis used two primary data sources:
1. **Disbursements**: Loan disbursement data including customer ID, loan amount, fee, tenure, and disbursement date.
2. **Repayments**: Loan repayment data including customer ID, repayment amount, repayment date, and repayment type.

## Python Code Structure

The analysis was implemented in Python with the following main components:

### 1. Data Loading and Cleaning
### 2. Loan Performance Data Preparation
### 3. Product Change Recommendations
### 4. Impact Quantification
### 5. Report Generation
### 6. Visualization Creation

## Data Transformation Steps
- Date Parsing: Custom parsing for Oracle-style dates in the repayments data.
- Tenure Extraction: Extracting numeric values from tenure strings.
- Expected Repayment Calculation: Adding tenure days to disbursement date.
- Default Identification: Identifying loans past due with outstanding balance.
- Profit Calculation: Calculating profit as total repaid minus loan amount.
- Segmentation: Binning loans by amount and tenure for analysis.

## Assumptions and Limitations
- Customer Matching: Repayments are matched to disbursements by customer ID.
- Default Definition: Loans are considered defaulted if past due with outstanding balance.
- Impact Estimates: Impact calculations use conservative assumptions about customer behavior.
- Data Completeness: Analysis assumes the provided data is representative of the full loan portfolio.

## Conclusion

This code implements a comprehensive analysis of loan product performance and generates data-driven recommendations for product enhancements. The modular structure allows for easy updates and extensions as new data becomes available or business requirements evolve.
"""

    # Save the documentation to a file
    with open("code_documentation.md", "w") as f:
        f.write(documentation)

    print("Code documentation generated and saved as 'code_documentation.md'")
    return documentation

# ===== MAIN EXECUTION =====

def main():
    try:
        # Step 0: Load and clean data
        disbursements_df, repayments_df = load_and_clean_data(excel_file)

        # Prepare loan performance data
        loan_status, repayment_types, repeat_rate = prepare_loan_performance_data(disbursements_df, repayments_df)

        # Step 1: Recommend data-driven product changes
        product_recommendations = recommend_product_changes(loan_status, repayment_types, repayments_df)

        # Step 2: Quantify potential impact of recommendations
        impact_metrics = quantify_impact(loan_status, product_recommendations,repayments_df)

        # Step 3: Compile comprehensive final report
        final_report = compile_final_report(loan_status, product_recommendations, impact_metrics)

        # Step 4: Prepare presentation-ready visualizations
        visualization_files = prepare_visualizations(loan_status, product_recommendations, impact_metrics)

        # Step 5: Document SQL queries and Python code
        code_documentation = document_code()

        print("\n=== PRODUCT ENHANCEMENT ANALYSIS COMPLETE ===")
        print("The following files have been generated:")
        print("1. product_enhancement_recommendations.md - Comprehensive final report")
        print("2. code_documentation.md - Documentation of SQL queries and Python code")
        print("3. Visualizations:")
        for name, file in visualization_files.items():
            print(f"   - {file} - {name.replace('_', ' ').title()}")

        print("\nCreated/Modified files during execution:")
        print("- product_enhancement_recommendations.md")
        print("- code_documentation.md")
        print("- fee_structure_comparison.html")
        print("- fee_structure_comparison.png")
        print("- tenure_options_visualization.html")
        print("- tenure_options_visualization.png")
        print("- loan_amount_risk_profitability.html")
        print("- loan_amount_risk_profitability.png")
        print("- repayment_methods_visualization.html")
        print("- repayment_methods_visualization.png")
        print("- impact_summary_visualization.html")
        print("- impact_summary_visualization.png")
        print("- implementation_roadmap.html")
        print("- implementation_roadmap.png")

    except Exception as e:
        print(f"Error during execution: {str(e)}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

=== PRODUCT ENHANCEMENT RECOMMENDATIONS AND FINAL REPORT ===

Loading and cleaning data...
- Disbursements: 26585 rows, 6 columns
- Repayments: 66016 rows, 5 columns
Data cleaning complete.

Preparing loan performance data...
Loan performance data prepared.

=== STEP 1: RECOMMEND DATA-DRIVEN PRODUCT CHANGES ===

1.1 Optimal Fee Structure Adjustments
Recommended Fee Structure Adjustments:
  loan_amount_bin  current_fee_pct  recommended_fee_pct                                          rationale
0           0-500            11.43                10.29  Decrease to improve competitiveness while main...
1        501-1000            11.96                10.76  Decrease to improve competitiveness while main...
2       1001-2000            13.06                11.75  Decrease to improve competitiveness while main...
3       2001-5000            13.38                12.04  Decrease to improve competitiveness while main...

1.2 Tenure Options Modifications
Recommended Tenure Options Modifications

### ==== STEP 5: DOCUMENT SQL QUERIES AND PYTHON CODE ====
```Python
def document_code():
    print("\n=== STEP 5: DOCUMENT SQL QUERIES AND PYTHON CODE ===")
    
    # Create documentation content
    documentation = """# Code Documentation: Product Enhancement Analysis
```
## Overview

This document provides documentation for the SQL queries and Python code used in the loan product enhancement analysis. The analysis was conducted to identify opportunities for optimizing the loan product offering based on historical performance data.

## Data Sources

The analysis used two primary data sources:
1. **Disbursements**: Loan disbursement data including customer ID, loan amount, fee, tenure, and disbursement date.
2. **Repayments**: Loan repayment data including customer ID, repayment amount, repayment date, and repayment type.

## Python Code Structure

The analysis was implemented in Python with the following main components:

### 1. Data Loading and Cleaning

```python
def load_and_clean_data(file_path):
    # Load disbursements and repayments data
    disbursements_raw = pd.read_excel(file_path, sheet_name='Disbursements')
    repayments_raw = pd.read_excel(file_path, sheet_name='Repayments')
    
    # Clean disbursements data
    # - Convert dates to datetime
    # - Extract numeric values from tenure
    # - Calculate expected repayment date and amount
    # - Calculate fee percentage
    
    # Clean repayments data
    # - Parse Oracle-style date format
    # - Ensure numeric repayment amounts
    
    return disbursements_df, repayments_df

2. Loan Performance Data Preparation
def prepare_loan_performance_data(disbursements_df, repayments_df):
    # Aggregate repayments by customer
    # Calculate outstanding amounts and repayment ratios
    # Identify defaulted loans
    # Calculate profit per loan
    # Define loan amount and tenure bins
    # Analyze repayment types
    # Calculate customer repeat rate
    
    return loan_status, repayment_types, repeat_rate

3. Product Change Recommendations
def recommend_product_changes(loan_status, repayment_types):
    # Analyze current fee structure
    # Recommend optimal fee adjustments
    # Analyze tenure options and recommend modifications
    # Analyze loan amount limits and recommend refinements
    # Analyze repayment methods and recommend incentives
    
    return product_recommendations

4. Impact Quantification
def quantify_impact(loan_status, product_recommendations):
    # Calculate baseline metrics
    # Estimate revenue impact of fee structure changes
    # Estimate impact of tenure options
    # Estimate impact of loan limit changes
    # Estimate impact of repayment incentives
    # Calculate risk reduction potential
    # Estimate customer experience improvements
    
    return impact_metrics

5. Report Generation
def compile_final_report(loan_status, product_recommendations, impact_metrics):
    # Create executive summary
    # Document methodology and assumptions
    # Present key findings
    # Detail recommendations and implementation roadmap
    
    return report

6. Visualization Creation
def prepare_visualizations(loan_status, product_recommendations, impact_metrics):
    # Create fee structure comparison visualization
    # Create tenure options visualization
    # Create loan amount risk-profitability visualization
    # Create repayment methods visualization
    # Create impact summary visualization
    # Create implementation roadmap visualization
    
    return visualization_files

SQL Queries

While this analysis primarily used pandas for data manipulation, equivalent SQL queries for key analyses are provided below:

1. Loan Performance Analysis
-- Calculate loan performance metrics
SELECT 
    d.customer_id,
    d.loan_amount,
    d.loan_fee,
    d.tenure_days,
    d.expected_repayment,
    COALESCE(SUM(r.repayment_amount), 0) AS total_repaid,
    d.expected_repayment - COALESCE(SUM(r.repayment_amount), 0) AS outstanding_amount,
    CASE WHEN d.expected_repayment_date < CURRENT_DATE AND 
              d.expected_repayment - COALESCE(SUM(r.repayment_amount), 0) > 0 
         THEN 1 ELSE 0 END AS is_defaulted,
    COALESCE(SUM(r.repayment_amount), 0) - d.loan_amount AS profit,
    (COALESCE(SUM(r.repayment_amount), 0) - d.loan_amount) / d.loan_amount * 100 AS profit_margin
FROM 
    disbursements d
LEFT JOIN 
    repayments r ON d.customer_id = r.customer_id
GROUP BY 
    d.customer_id, d.loan_amount, d.loan_fee, d.tenure_days, d.expected_repayment, d.expected_repayment_date;

2. Fee Structure Analysis
-- Analyze fee structure by loan amount segment
WITH loan_performance AS (
    -- Loan performance query from above
)
SELECT 
    loan_amount_bin,
    AVG(fee_percentage) AS avg_fee_pct,
    MEDIAN(fee_percentage) AS median_fee_pct,
    STDDEV(fee_percentage) AS std_fee_pct,
    AVG(profit_margin) AS avg_profit_margin,
    MEDIAN(profit_margin) AS median_profit_margin,
    AVG(is_defaulted) AS default_rate,
    COUNT(*) AS loan_count
FROM 
    loan_performance
GROUP BY 
    loan_amount_bin
ORDER BY 
    loan_amount_bin;

3. Tenure Analysis
-- Analyze performance by tenure
WITH loan_performance AS (
    -- Loan performance query from above
)
SELECT 
    tenure_bin,
    AVG(profit_margin) AS avg_profit_margin,
    MEDIAN(profit_margin) AS median_profit_margin,
    AVG(is_defaulted) AS default_rate,
    COUNT(*) AS loan_count,
    AVG(loan_amount) AS avg_loan_amount,
    AVG(tenure_days) AS avg_tenure_days
FROM 
    loan_performance
GROUP BY 
    tenure_bin
ORDER BY 
    tenure_bin;

4. Repayment Method Analysis
-- Analyze repayment methods
SELECT 
    r.repayment_type,
    SUM(r.repayment_amount) AS total_repaid,
    COUNT(DISTINCT r.customer_id) AS unique_customers,
    AVG(lp.is_defaulted) AS default_rate
FROM 
    repayments r
LEFT JOIN (
    -- Loan performance query from above
) lp ON r.customer_id = lp.customer_id
GROUP BY 
    r.repayment_type;

5. Customer Repeat Rate Analysis
-- Calculate customer repeat rate
WITH customer_loan_counts AS (
    SELECT 
        customer_id,
        COUNT(*) AS loan_count
    FROM 
        disbursements
    GROUP BY 
        customer_id
)
SELECT 
    COUNT(CASE WHEN loan_count > 1 THEN 1 END) AS repeat_customers,
    COUNT(*) AS total_customers,
    COUNT(CASE WHEN loan_count > 1 THEN 1 END) * 100.0 / COUNT(*) AS repeat_rate
FROM 
    customer_loan_counts;

6. Loan Amount Limits Analysis
-- Analyze performance by loan amount segment
WITH loan_performance AS (
    -- Loan performance query from above
)
SELECT 
    loan_amount_bin,
    AVG(profit_margin) AS avg_profit_margin,
    AVG(is_defaulted) AS default_rate,
    COUNT(*) AS loan_count,
    AVG(loan_amount) AS avg_loan_amount,
    MIN(loan_amount) AS min_loan_amount,
    MAX(loan_amount) AS max_loan_amount
FROM 
    loan_performance
GROUP BY 
    loan_amount_bin
ORDER BY 
    loan_amount_bin;

7. Impact Estimation Query
-- Estimate impact of fee structure changes
WITH current_fees AS (
    SELECT 
        loan_amount_bin,
        AVG(fee_percentage) AS current_fee_pct,
        SUM(loan_amount) AS bin_loan_amount
    FROM 
        loan_performance
    GROUP BY 
        loan_amount_bin
),
recommended_fees AS (
    -- This would be a table with recommended fee percentages
    SELECT 
        loan_amount_bin,
        recommended_fee_pct
    FROM 
        fee_recommendations
)
SELECT 
    cf.loan_amount_bin,
    cf.current_fee_pct,
    rf.recommended_fee_pct,
    cf.bin_loan_amount,
    cf.bin_loan_amount * (rf.recommended_fee_pct - cf.current_fee_pct) / 100 AS fee_impact
FROM 
    current_fees cf
JOIN 
    recommended_fees rf ON cf.loan_amount_bin = rf.loan_amount_bin;

Data Transformation Steps
Date Parsing: Custom parsing for Oracle-style dates in the repayments data.
Tenure Extraction: Extracting numeric values from tenure strings.
Expected Repayment Calculation: Adding tenure days to disbursement date.
Default Identification: Identifying loans past due with outstanding balance.
Profit Calculation: Calculating profit as total repaid minus loan amount.
Segmentation: Binning loans by amount and tenure for analysis.
Fee Percentage Calculation: Calculating fee as a percentage of loan amount.
Repayment Ratio: Calculating the ratio of amount repaid to expected repayment.
Days Past Due: Calculating days between expected repayment date and current date.
Risk Scoring: Combining default rate, profit margin, and other factors to assess risk.
Assumptions and Limitations
Customer Matching: Repayments are matched to disbursements by customer ID.
Default Definition: Loans are considered defaulted if past due with outstanding balance.
Impact Estimates: Impact calculations use conservative assumptions about customer behavior.
Data Completeness: Analysis assumes the provided data is representative of the full loan portfolio.
Fee Structure: Assumes that fee percentages can be adjusted without significant market impact.
Tenure Options: Assumes that new tenure options can be implemented with existing systems.
Customer Behavior: Assumes that customer behavior will respond predictably to incentives.
Time Horizon: Impact estimates are based on a one-year projection period.
Best Practices Implemented
Modular Code Structure: Functions with clear responsibilities for maintainability.
Error Handling: Try-except blocks to catch and report errors.
Data Validation: Checking for missing values and data type consistency.
Documentation: Comprehensive comments and documentation.
Visualization: Clear, informative visualizations with proper labels and titles.
Report Generation: Structured report with executive summary and detailed findings.
Performance Optimization: Efficient data processing with appropriate indexing.
Code Reusability: Generic functions that can be applied to different datasets.
Conclusion

This code implements a comprehensive analysis of loan product performance and generates data-driven recommendations for product enhancements. The modular structure allows for easy updates and extensions as new data becomes available or business requirements evolve.

The analysis provides actionable insights for optimizing fee structure, tenure options, loan amount limits, and repayment incentives, with quantified estimates of the potential impact on revenue, risk, and customer experience.

Future Enhancements

Machine Learning Models: Implement predictive models for default risk and customer behavior.

A/B Testing Framework: Add capability to design and analyze A/B tests for product changes.

Real-time Monitoring: Develop real-time dashboards to monitor impact of implemented changes.

Customer Segmentation: Enhance analysis with more sophisticated customer segmentation.

Competitive Analysis: Incorporate data on competitor offerings for market positioning. """

Save the documentation to a file

with open("code_documentation.md", "w") as f: f.write(documentation)

print("Code documentation generated and saved as 'code_documentation.md'") return documentation


This function creates comprehensive documentation for all the code and SQL queries used in the analysis. The documentation includes:

1. **Overview**: A high-level description of the analysis purpose
2. **Data Sources**: Description of the primary data sources used
3. **Python Code Structure**: Breakdown of the main functions with explanations
4. **SQL Queries**: Equivalent SQL queries for key analyses
5. **Data Transformation Steps**: Detailed explanation of data processing steps
6. **Assumptions and Limitations**: Important caveats and assumptions
7. **Best Practices Implemented**: Software engineering best practices used
8. **Future Enhancements**: Suggestions for further development

The documentation is saved as a Markdown file, making it easy to read and share with stakeholders. This comprehensive documentation ensures that the analysis is transparent, reproducible, and maintainable by other team members.
